# 02 — EPIC-KITCHENS-100 Dataset Statistics (DataLoader-Only)

This notebook computes comprehensive dataset statistics for **EPIC-KITCHENS-100** using **exclusively the unified PyTorch `DataLoader`** (`EgoClipDataset.from_config` + `torch.utils.data.DataLoader`). No raw dataset files, index JSONLs, or adapters are accessed directly.

### What EPIC-KITCHENS-100 provides through `DataLoader`
- **Action-Centred Clips (`sampling="actions"`):** one clip centred on every narrated action segment (`action` narration string, `timestamps`, `frame_indices`, `sequence_id`).
- **GoPro Camera-Frame IMU (`imu`):** ~200 Hz accelerometer + gyroscope interpolated to clip frames (`imu`, `imu_valid`), verified in the OpenCV camera frame (`imu_in_camera_frame == True`) with unit gravity vector (`gravity_cam`).
- **RGB Video & Audio (`rgb`, `audio`):** egocentric head-mounted GoPro RGB video and 16 kHz resampled mono audio.
- *(No 3D hand/object/camera GT: `K_valid`, `hand_valid`, `obj_valid`, and `cam_valid` are expected to be `False`.)*

### Two-Pass DataLoader Strategy
1. **Pass 1 — Full-Dataset Action & IMU Scan via `DataLoader`** (`sampling="actions"`, `modalities=["imu"]`): iterates **all** action-centred clips across all local participants (`P01`–`P05`) without video decoding, analyzing participant/video/action distributions, narration vocabulary, GoPro IMU dynamics, and head pitch/roll tilt from `gravity_cam`.
2. **Pass 2 — Dense Multimodal `DataLoader`** (`modalities=["rgb", "audio", "imu"]`, `256×256` transform): measures RGB intensity, visual temporal motion vs. IMU angular velocity correlation, audio RMS/peak loudness, and renders synchronized RGB + audio + IMU examples from batch tensors.

In [ ]:
import re
import time
import warnings
from collections import Counter, defaultdict

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

from robot.data.torch_dataset import EgoClipDataset

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

# --- Notebook Configuration ---
SEED = 42
PASS1_MAX_CLIPS = None   # None = scan 100% of EPIC action clips (~24k clips, fast without RGB)
PASS2_MAX_CLIPS = 96     # Dense RGB + Audio + IMU clips
BATCH_SIZE_P1 = 128
BATCH_SIZE_P2 = 8
NUM_WORKERS = 8

def make_loader(cfg: dict, max_clips: int | None, batch_size: int, seed: int = SEED) -> tuple[DataLoader, int]:
    ds = EgoClipDataset.from_config(cfg)
    total = len(ds)
    if max_clips is not None and total > max_clips:
        rng = np.random.default_rng(seed)
        idx = np.sort(rng.choice(total, size=max_clips, replace=False)).tolist()
        ds_view = Subset(ds, idx)
    else:
        ds_view = ds
    dl = DataLoader(ds_view, batch_size=batch_size, shuffle=False, num_workers=NUM_WORKERS)
    return dl, total

print("Setup complete. PyTorch:", torch.__version__)

## 1. Pass 1 — Full-Dataset Action & GoPro IMU Scan via `DataLoader`

In [ ]:
cfg_pass1 = {
    "clip_len": 16,
    "fps": 15,
    "max_objects": 4,
    "modalities": ["imu"],
    "sources": [
        {
            "dataset": "epic_kitchens",
            "sampling": "actions",
        }
    ],
}

t0 = time.time()
dl_p1, total_windows_p1 = make_loader(cfg_pass1, PASS1_MAX_CLIPS, BATCH_SIZE_P1)
print(f"Pass 1 DataLoader ready in {time.time() - t0:.2f}s | total action clips: {total_windows_p1:,} | iterating: {len(dl_p1.dataset):,}")

clip_rows = []
narration_counter = Counter()
verb_counter = Counter()
noun_counter = Counter()

accel_samples = []
gyro_samples = []
gravity_samples = []
verb_gyro = defaultdict(list)

# Track exact absence of 3D labels
gt_flags = Counter()
total_clips = 0

t_scan = time.time()
for batch in dl_p1:
    B, T = batch["timestamps"].shape
    total_clips += B
    ts = batch["timestamps"].numpy()
    fi = batch["frame_indices"].numpy()
    imu = batch["imu"].numpy()                          # (B, T, 6) [ax, ay, az, gx, gy, gz]
    imu_v = batch["imu_valid"].numpy()                  # (B, T)
    imu_cam = batch["imu_in_camera_frame"].numpy()      # (B,)
    gcam = batch["gravity_cam"].numpy()                 # (B, 3)

    gt_flags["K_valid"] += int(batch["K_valid"].sum())
    gt_flags["hand_valid"] += int(batch["hand_valid"].any(dim=(1, 2)).sum())
    gt_flags["obj_valid"] += int(batch["obj_valid"].any(dim=(1, 2)).sum())
    gt_flags["cam_valid"] += int(batch["cam_valid"].any(dim=1).sum())

    # Estimate native video fps from dframe / dt
    dt = np.maximum(ts[:, -1] - ts[:, 0], 1e-6)
    est_native_fps = (fi[:, -1] - fi[:, 0]) / dt

    for b in range(B):
        sid = batch["sequence_id"][b]
        participant = sid.split("_")[0]
        act = (batch["action"][b] or "").strip().lower()
        narration_counter[act] += 1
        toks = [w for w in re.split(r"[^a-z]+", act) if w]
        lead_verb = toks[0] if toks else "(empty)"
        lead_noun = toks[-1] if len(toks) >= 2 else "(none)"
        verb_counter[lead_verb] += 1
        if lead_noun != "(none)":
            noun_counter[lead_noun] += 1

        iv = imu_v[b]
        has_imu = bool(iv.any())
        a_norm = float(np.linalg.norm(imu[b, iv, :3], axis=-1).mean()) if has_imu else np.nan
        g_norm = float(np.linalg.norm(imu[b, iv, 3:], axis=-1).mean()) if has_imu else np.nan

        if has_imu:
            verb_gyro[lead_verb].append(g_norm)
            if (b % 4) == 0:  # lightweight subsample of raw per-frame vectors
                accel_samples.append(imu[b, iv, :3])
                gyro_samples.append(imu[b, iv, 3:])
            if imu_cam[b] and np.linalg.norm(gcam[b]) > 0.5:
                gravity_samples.append(gcam[b])

        clip_rows.append({
            "sequence_id": sid,
            "participant": participant,
            "center_s": float(0.5 * (ts[b, 0] + ts[b, -1])),
            "end_s": float(ts[b, -1]),
            "est_native_fps": float(est_native_fps[b]),
            "action": act,
            "lead_verb": lead_verb,
            "n_words": len(toks),
            "imu_valid_frac": float(iv.mean()),
            "imu_in_cam": bool(imu_cam[b]),
            "accel_norm": a_norm,
            "gyro_norm": g_norm,
        })

df = pd.DataFrame(clip_rows)
print(f"Pass 1 completed in {time.time() - t_scan:.2f}s | {len(df):,} clips across {df['sequence_id'].nunique()} videos")

## 2. Participant, Video & Recording Rate Summary

In [ ]:
vid_df = df.groupby(["participant", "sequence_id"]).agg(
    clips=("action", "count"),
    approx_duration_min=("end_s", lambda s: s.max() / 60.0),
    native_fps=("est_native_fps", "median"),
    has_imu=("imu_valid_frac", lambda s: (s > 0).any()),
    imu_in_cam=("imu_in_cam", "any"),
).reset_index()
vid_df["actions_per_min"] = vid_df["clips"] / np.maximum(vid_df["approx_duration_min"], 0.1)

part_df = vid_df.groupby("participant").agg(
    videos=("sequence_id", "count"),
    videos_with_imu=("has_imu", "sum"),
    total_action_clips=("clips", "sum"),
    total_hours=("approx_duration_min", lambda s: s.sum() / 60.0),
    median_actions_per_min=("actions_per_min", "median"),
).reset_index()

summary = pd.DataFrame([
    ("Total action clips analyzed", f"{len(df):,}"),
    ("Unique videos / participants", f"{vid_df['sequence_id'].nunique()} / {vid_df['participant'].nunique()} ({', '.join(sorted(df['participant'].unique()))})"),
    ("Approx. total annotated footage span", f"{vid_df['approx_duration_min'].sum()/60.0:.2f} hours"),
    ("Median video span / actions per video", f"{vid_df['approx_duration_min'].median():.1f} min / {vid_df['clips'].median():.0f} actions"),
    ("Median action density", f"{vid_df['actions_per_min'].median():.1f} actions/min"),
    ("Videos with GoPro IMU (camera frame)", f"{int(vid_df['has_imu'].sum())}/{len(vid_df)} ({100*vid_df['has_imu'].mean():.1f}%) | cam-frame: {int(vid_df['imu_in_cam'].sum())}"),
    ("Clips with valid IMU frames", f"{(df['imu_valid_frac'] > 0).mean()*100:.1f}% (frame-level: {df['imu_valid_frac'].mean()*100:.1f}%)"),
    ("Unique narration strings", f"{len(narration_counter):,}"),
    ("3D GT flags (K / hand / obj / cam)", f"{gt_flags['K_valid']} / {gt_flags['hand_valid']} / {gt_flags['obj_valid']} / {gt_flags['cam_valid']} (uncalibrated in-the-wild)"),
], columns=["Metric", "Value"])
display(summary)
display(part_df.round(2))

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
axes[0].bar(part_df["participant"], part_df["total_action_clips"], color="#1b9e77")
axes[0].set_title("Action Clips per Participant")
axes[0].set_ylabel("Action clips")

axes[1].hist(vid_df["approx_duration_min"], bins=25, color="#7570b3")
axes[1].axvline(vid_df["approx_duration_min"].median(), color="red", ls="--", label=f"Med={vid_df['approx_duration_min'].median():.1f} min")
axes[1].set_title("Video Duration Distribution (from Max Clip Timestamp)")
axes[1].set_xlabel("Duration (minutes)")
axes[1].legend()

axes[2].hist(vid_df["native_fps"].clip(40, 65), bins=25, color="#d95f02")
axes[2].set_title("Estimated Native Video FPS (50 vs. 59.94 fps GoPro)")
axes[2].set_xlabel("Native FPS (dframe / dt)")
plt.tight_layout()
plt.show()

## 3. Narration Vocabulary, Long-Tail Zipf Curve & Top Verbs/Nouns

In [ ]:
top_narr = pd.DataFrame(narration_counter.most_common(20), columns=["narration", "clips"])
top_narr["share_%"] = 100.0 * top_narr["clips"] / len(df)
display(top_narr.head(12).round(2))

counts_sorted = np.array([c for _, c in narration_counter.most_common()], dtype=np.float64)
cum_cov = np.cumsum(counts_sorted) / counts_sorted.sum() * 100.0

fig, axes = plt.subplots(1, 3, figsize=(16, 4.8))

# Top-15 narrations
tn = top_narr.head(15).iloc[::-1]
axes[0].barh(tn["narration"], tn["clips"], color="#2b8cbe")
axes[0].set_title("Top-15 Action Narrations")
axes[0].set_xlabel("Clips")

# Top-15 leading verbs
tv = pd.DataFrame(verb_counter.most_common(15), columns=["verb", "clips"]).iloc[::-1]
axes[1].barh(tv["verb"], tv["clips"], color="#e66101")
axes[1].set_title("Top-15 Leading Action Verbs")
axes[1].set_xlabel("Clips")

# Zipf cumulative coverage
axes[2].plot(np.arange(1, len(cum_cov) + 1), cum_cov, lw=2, color="#5e3c99")
axes[2].set_xscale("log")
for k in (50, 200, 1000):
    if k < len(cum_cov):
        axes[2].axvline(k, ls=":", color="gray")
        axes[2].text(k * 1.1, cum_cov[k - 1] - 4, f"Top-{k}: {cum_cov[k-1]:.1f}%", fontsize=8)
axes[2].set_title("Narration Vocabulary Cumulative Coverage")
axes[2].set_xlabel("Top-K unique narrations (log scale)")
axes[2].set_ylabel("Cumulative clip coverage (%)")

plt.tight_layout()
plt.show()

## 4. GoPro Camera-Frame IMU & Head-Tilt Gravity Analysis

In EPIC-KITCHENS, the GoPro `gpmd` telemetry is mapped to the OpenCV camera frame (`x` right, `y` down, `z` forward). Specific force at rest points **up** (`-y`), while `gravity_cam` is the unit **down** vector (`-mean(accel) / ||mean(accel)||`). From `gravity_cam = (gx, gy, gz)` we directly compute the wearer's downward head pitch (`atan2(gz, gy)`) and lateral head roll (`atan2(gx, gy)`).

In [ ]:
acc = np.concatenate(accel_samples) if accel_samples else np.zeros((0, 3))
gyr = np.concatenate(gyro_samples) if gyro_samples else np.zeros((0, 3))
gvec = np.array(gravity_samples) if gravity_samples else np.zeros((0, 3))

acc_mag = np.linalg.norm(acc, axis=-1)
gyr_mag = np.linalg.norm(gyr, axis=-1)
# Pitch: angle camera optical axis (+z) tilts downward below horizon = atan2(gz, gy)
pitch_deg = np.degrees(np.arctan2(gvec[:, 2], gvec[:, 1])) if len(gvec) else np.zeros(0)
roll_deg = np.degrees(np.arctan2(gvec[:, 0], gvec[:, 1])) if len(gvec) else np.zeros(0)

imu_stats = pd.DataFrame([
    ("Mean accel vector (OpenCV x,y,z) m/s²", f"({acc[:,0].mean():.2f}, {acc[:,1].mean():.2f}, {acc[:,2].mean():.2f})"),
    ("Accel magnitude ||a|| [p10 / med / p90]", f"{np.percentile(acc_mag, 10):.2f} / {np.median(acc_mag):.2f} / {np.percentile(acc_mag, 90):.2f} m/s²"),
    ("Gyro angular speed ||ω|| [med / p90 / p99]", f"{np.median(gyr_mag):.3f} / {np.percentile(gyr_mag, 90):.3f} / {np.percentile(gyr_mag, 99):.3f} rad/s"),
    ("Mean gravity_cam unit vector (down)", f"({gvec[:,0].mean():.3f}, {gvec[:,1].mean():.3f}, {gvec[:,2].mean():.3f})"),
    ("Inferred downward head pitch [p10 / med / p90]", f"{np.percentile(pitch_deg, 10):.1f}° / {np.median(pitch_deg):.1f}° / {np.percentile(pitch_deg, 90):.1f}°"),
    ("Inferred head roll [p10 / med / p90]", f"{np.percentile(roll_deg, 10):.1f}° / {np.median(roll_deg):.1f}° / {np.percentile(roll_deg, 90):.1f}°"),
], columns=["GoPro IMU Metric", "Value"])
display(imu_stats)

fig, axes = plt.subplots(2, 2, figsize=(13, 8.2))

# 1. Accel magnitude
axes[0, 0].hist(np.clip(acc_mag, 5, 16), bins=60, color="#1b9e77")
axes[0, 0].axvline(9.81, color="red", ls="--", label="1 g (9.81 m/s²)")
axes[0, 0].set_title("Accelerometer Magnitude ||a|| Distribution")
axes[0, 0].set_xlabel("||a|| (m/s²)")
axes[0, 0].legend()

# 2. Gyro magnitude
axes[0, 1].hist(np.clip(gyr_mag, 0, 3.0), bins=60, color="#d95f02")
axes[0, 1].axvline(np.median(gyr_mag), color="black", ls="--", label=f"Med={np.median(gyr_mag):.2f} rad/s")
axes[0, 1].set_title("Gyroscope Angular Speed ||ω|| Distribution")
axes[0, 1].set_xlabel("||ω|| (rad/s)")
axes[0, 1].legend()

# 3. Head pitch & roll from gravity_cam
axes[1, 0].hist(np.clip(pitch_deg, -30, 85), bins=50, alpha=0.75, color="#2b8cbe", label=f"Pitch (med={np.median(pitch_deg):.1f}°)")
axes[1, 0].hist(np.clip(roll_deg, -45, 45), bins=50, alpha=0.6, color="#e7298a", label=f"Roll (med={np.median(roll_deg):.1f}°)")
axes[1, 0].set_title("Head Orientation Relative to Gravity (from gravity_cam)")
axes[1, 0].set_xlabel("Angle (degrees)")
axes[1, 0].legend()

# 4. Head motion intensity by top verbs
v_motion = [
    (v, np.median(verb_gyro[v]), len(verb_gyro[v]))
    for v, _ in verb_counter.most_common(15)
    if len(verb_gyro[v]) >= 50
]
v_motion.sort(key=lambda x: x[1])
axes[1, 1].barh([x[0] for x in v_motion], [x[1] for x in v_motion], color="#7570b3")
axes[1, 1].set_title("Median Head Angular Speed ||ω|| by Top Verb")
axes[1, 1].set_xlabel("Median ||ω|| (rad/s)")

plt.tight_layout()
plt.show()

## 5. Pass 2 — Dense Multimodal `DataLoader` (RGB + Audio + IMU)

In [ ]:
cfg_pass2 = {
    "clip_len": 8,
    "fps": 15,
    "max_objects": 4,
    "modalities": ["rgb", "audio", "imu"],
    "audio": {"sample_rate": 16000, "mono": True},
    "transforms": {"resize": 256, "center_crop": [256, 256]},
    "sources": [
        {
            "dataset": "epic_kitchens",
            "sampling": "actions",
        }
    ],
}

t0 = time.time()
dl_p2, _ = make_loader(cfg_pass2, PASS2_MAX_CLIPS, BATCH_SIZE_P2, seed=SEED + 1)

rgb_sum = np.zeros(3, np.float64)
rgb_sq = np.zeros(3, np.float64)
rgb_n = 0
frame_diff_list = []
clip_gyro_list = []
audio_rms_db = []
audio_peak = []
audio_zcr = []
audio_valid_cnt = 0
vis_samples = []

for batch in dl_p2:
    frames = batch["frames"].numpy()          # (B, T, 3, H, W)
    audio = batch["audio"].numpy()            # (B, 1, S)
    a_val = batch["audio_valid"].numpy()
    imu = batch["imu"].numpy()                # (B, T, 6)
    i_val = batch["imu_valid"].numpy()

    B, T = frames.shape[:2]
    rgb_sum += frames.sum(axis=(0, 1, 3, 4))
    rgb_sq += (frames ** 2).sum(axis=(0, 1, 3, 4))
    rgb_n += B * T * frames.shape[3] * frames.shape[4]

    fdiff = np.abs(np.diff(frames, axis=1)).mean(axis=(1, 2, 3, 4))  # (B,)
    for b in range(B):
        frame_diff_list.append(float(fdiff[b]))
        if i_val[b].any():
            clip_gyro_list.append((float(fdiff[b]), float(np.linalg.norm(imu[b, i_val[b], 3:], axis=-1).mean())))
        if a_val[b]:
            audio_valid_cnt += 1
            wav = audio[b, 0]
            rms = float(np.sqrt(np.mean(wav ** 2) + 1e-12))
            audio_rms_db.append(20.0 * np.log10(rms))
            audio_peak.append(float(np.abs(wav).max()))
            audio_zcr.append(float(np.mean(wav[:-1] * wav[1:] < 0)))

        if len(vis_samples) < 4 and a_val[b] and i_val[b].all():
            vis_samples.append({
                "seq_id": batch["sequence_id"][b],
                "action": batch["action"][b],
                "rgb": np.transpose(frames[b, T // 2], (1, 2, 0)),
                "wav": audio[b, 0],
                "imu": imu[b],
                "ts": batch["timestamps"][b].numpy() - float(batch["timestamps"][b, 0]),
            })

rgb_mean = rgb_sum / rgb_n
rgb_std = np.sqrt(np.maximum(rgb_sq / rgb_n - rgb_mean ** 2, 0.0))
fg = np.array(clip_gyro_list) if clip_gyro_list else np.zeros((0, 2))
corr = float(np.corrcoef(fg[:, 0], fg[:, 1])[0, 1]) if len(fg) > 2 else np.nan

dense_summary = pd.DataFrame([
    ("Pass 2 runtime / clips", f"{time.time() - t0:.1f}s / {len(dl_p2.dataset)} clips"),
    ("RGB channel mean (R, G, B)", f"({rgb_mean[0]:.3f}, {rgb_mean[1]:.3f}, {rgb_mean[2]:.3f})"),
    ("RGB channel std (R, G, B)", f"({rgb_std[0]:.3f}, {rgb_std[1]:.3f}, {rgb_std[2]:.3f})"),
    ("Mean temporal frame diff |I_t - I_{t-1}|", f"{np.mean(frame_diff_list):.4f}"),
    ("Visual motion vs. IMU ||ω|| Pearson r", f"{corr:.3f} (n={len(fg)})"),
    ("Audio valid rate", f"{audio_valid_cnt}/{len(dl_p2.dataset)} ({100*audio_valid_cnt/len(dl_p2.dataset):.1f}%)"),
    ("Audio RMS loudness [p10 / med / p90]", f"{np.percentile(audio_rms_db, 10):.1f} / {np.median(audio_rms_db):.1f} / {np.percentile(audio_rms_db, 90):.1f} dBFS"),
    ("Audio peak amplitude [med / p95]", f"{np.median(audio_peak):.3f} / {np.percentile(audio_peak, 95):.3f}"),
], columns=["Dense Modality Metric", "Value"])
display(dense_summary)

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))
if len(fg):
    axes[0].scatter(fg[:, 1], fg[:, 0], alpha=0.65, color="#1b9e77")
    axes[0].set_title(f"Visual Frame Diff vs. GoPro Gyro Speed (r = {corr:.2f})")
    axes[0].set_xlabel("Clip mean IMU ||ω|| (rad/s)")
    axes[0].set_ylabel("Mean |I_t - I_{t-1}|")

axes[1].hist(audio_rms_db, bins=30, color="#7570b3")
axes[1].axvline(np.median(audio_rms_db), color="red", ls="--", label=f"Med={np.median(audio_rms_db):.1f} dBFS")
axes[1].set_title("Audio Clip RMS Loudness Distribution (16 kHz Mono)")
axes[1].set_xlabel("RMS (dBFS)")
axes[1].legend()
plt.tight_layout()
plt.show()

## 6. Visual Sanity Check: Synchronized RGB + Audio + GoPro IMU from `DataLoader`

In [ ]:
if vis_samples:
    fig, axes = plt.subplots(len(vis_samples), 3, figsize=(13, 3.1 * len(vis_samples)))
    if len(vis_samples) == 1:
        axes = axes[None, :]
    for row, s in enumerate(vis_samples):
        axes[row, 0].imshow(s["rgb"])
        axes[row, 0].set_title(f"{s['seq_id']} | {s['action']}", fontsize=9)
        axes[row, 0].axis("off")

        t_wav = np.linspace(0, len(s["wav"]) / 16000.0, len(s["wav"]))
        axes[row, 1].plot(t_wav, s["wav"], lw=0.6, color="#2b8cbe")
        axes[row, 1].set_ylim(-1, 1)
        axes[row, 1].set_title("Audio Waveform (16 kHz)", fontsize=9)
        axes[row, 1].set_xlabel("Time (s)")

        axes[row, 2].plot(s["ts"], s["imu"][:, 0], label="ax")
        axes[row, 2].plot(s["ts"], s["imu"][:, 1], label="ay")
        axes[row, 2].plot(s["ts"], s["imu"][:, 2], label="az")
        axes[row, 2].set_title("GoPro Accel (OpenCV cam frame, m/s²)", fontsize=9)
        axes[row, 2].set_xlabel("Time (s)")
        if row == 0:
            axes[row, 2].legend(fontsize=7, loc="upper right")
    plt.tight_layout()
    plt.show()